# Sentence-only intervention (layers 15 and 23)

Этот ноутбук работает **только** в режиме `sentence` и использует файлы из `checkpoints/`.

Что делает:
1) удобно загружает `test.csv` и `Hs_hedge_universal.pt` в Colab при необходимости,
2) вешает интервенцию на слои 15 и 23,
3) делает smoke test,
4) делает full run по всему `test.csv`.


In [ ]:
# If needed in Colab:
# !pip -q install transformers accelerate sentencepiece pandas tqdm

import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
# -----------------------
# Config (sentence-only)
# -----------------------
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
DEVICE_MAP = "auto"
DTYPE = torch.float16

LAYER_IDS = [15, 23]
FIXED_ALPHA = 0.7

USE_ALPHA_QUANTIZATION = True
ALPHA_QUANT_STEP = 0.4

SMOKE_BATCH_SIZE = 2
FULL_BATCH_SIZE = 40
MAX_NEW_TOKENS = 64

ROOT = Path.cwd().resolve().parent

DATA_PATH = ROOT / "checkpoints" / "datasets__nq_open" / "Mistral-7B-Instruct-v0.3_sentence" / "test.csv"
ALPHA_SOURCE_PATH = ROOT / "checkpoints" / "datasets__nq_open" / "Mistral-7B-Instruct-v0.3" / "test.csv"
HEDGE_PATH = ROOT / "checkpoints" / "calibration" / "outputs" / "merged" / "Mistral-7B-Instruct-v0.3" / "uncertainty" / "Hs_hedge_universal.pt"

MAX_SE = 2.302585092994045
MAX_ALPHA = 1.6

OUT_DIR = ROOT / "checkpoints" / "intervention_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

qstep_tag = str(ALPHA_QUANT_STEP).replace('.', 'p') if USE_ALPHA_QUANTIZATION else 'none'
SMOKE_OUT = OUT_DIR / f"smoke_sentence_layers15_23_q{qstep_tag}.jsonl"
FULL_OUT = OUT_DIR / f"full_sentence_layers15_23_q{qstep_tag}.jsonl"

print("DATA_PATH:", DATA_PATH)
print("ALPHA_SOURCE_PATH:", ALPHA_SOURCE_PATH)
print("HEDGE_PATH:", HEDGE_PATH)
print("SMOKE_OUT:", SMOKE_OUT)
print("FULL_OUT:", FULL_OUT)


DATA_PATH: /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv
ALPHA_SOURCE_PATH: /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3/test.csv
HEDGE_PATH: /checkpoints/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt
SMOKE_OUT: /checkpoints/intervention_outputs/smoke_sentence_layers15_23_q0p4.jsonl
FULL_OUT: /checkpoints/intervention_outputs/full_sentence_layers15_23_q0p4.jsonl


## Colab helper: upload all required files

Проверяет наличие и при необходимости просит загрузить все обязательные файлы в правильные пути внутри `checkpoints/`.


In [ ]:
def _is_valid_uploaded_payload(path_obj, payload_bytes, human_name):
    name = path_obj.name.lower()
    if name.endswith('.csv'):
        try:
            text_head = payload_bytes[:2048].decode('utf-8', errors='ignore')
        except Exception:
            return False, f"{human_name}: cannot decode CSV header"
        if ',' not in text_head:
            return False, f"{human_name}: uploaded file does not look like CSV"
        return True, "ok"

    if name.endswith('.pt'):
        tmp = path_obj.parent / (path_obj.name + '.tmp_validate')
        try:
            with open(tmp, 'wb') as f:
                f.write(payload_bytes)
            obj = torch.load(tmp, map_location='cpu')
            if not torch.is_tensor(obj):
                return False, f"{human_name}: .pt loaded, but object is not a tensor"
            if obj.ndim != 2:
                return False, f"{human_name}: expected 2D tensor, got ndim={obj.ndim}"
            return True, "ok"
        except Exception as e:
            return False, f"{human_name}: cannot load as torch .pt ({e})"
        finally:
            if tmp.exists():
                tmp.unlink()

    return True, "ok"


def _upload_to_path_if_missing(path_obj, human_name, hint_path):
    if path_obj.exists():
        print(f"OK: {human_name} already exists -> {path_obj}")
        return

    print(f"Missing: {human_name}")
    print("Expected path:")
    print("-", path_obj)
    if hint_path:
        print("Where to find in project:")
        print("-", hint_path)

    try:
        from google.colab import files
    except Exception:
        raise FileNotFoundError(
            f"Not in Colab and missing file: {path_obj}. Copy it manually and rerun."
        )

    path_obj.parent.mkdir(parents=True, exist_ok=True)

    while True:
        print("Open upload dialog and choose the file now...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError(f"No file uploaded for: {human_name}")

        first_name = next(iter(uploaded.keys()))
        payload = uploaded[first_name]

        ok, reason = _is_valid_uploaded_payload(path_obj, payload, human_name)
        if not ok:
            print(f"Invalid uploaded file: {first_name}")
            print("Reason:", reason)
            print("Please upload the correct file again.\n")
            continue

        with open(path_obj, "wb") as f:
            f.write(payload)

        print(f"Saved {human_name} to: {path_obj}")
        print("Uploaded filename:", first_name)
        break


def ensure_required_files_uploaded():
    required = [
        (
            DATA_PATH,
            "sentence test.csv",
            "checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv",
        ),
        (
            ALPHA_SOURCE_PATH,
            "alpha source test.csv",
            "checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3/test.csv",
        ),
        (
            HEDGE_PATH,
            "Hs_hedge_universal.pt",
            "checkpoints/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt",
        ),
    ]

    for path_obj, human_name, hint in required:
        _upload_to_path_if_missing(path_obj, human_name, hint)

    print("\nAll required files are ready.")


ensure_required_files_uploaded()


Missing: sentence test.csv
Expected path:
- /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv
Where to find in project:
- checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv
Open upload dialog and choose the file now...


Saving test.csv to test.csv
Saved sentence test.csv to: /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv
Uploaded filename: test.csv
Missing: alpha source test.csv
Expected path:
- /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3/test.csv
Where to find in project:
- checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3/test.csv
Open upload dialog and choose the file now...


Saving test_new.csv to test_new.csv
Saved alpha source test.csv to: /checkpoints/datasets__nq_open/Mistral-7B-Instruct-v0.3/test.csv
Uploaded filename: test_new.csv
Missing: Hs_hedge_universal.pt
Expected path:
- /checkpoints/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt
Where to find in project:
- checkpoints/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt
Open upload dialog and choose the file now...


Saving Hs_hedge_universal.pt to Hs_hedge_universal.pt
Saved Hs_hedge_universal.pt to: /checkpoints/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt
Uploaded filename: Hs_hedge_universal.pt

All required files are ready.


In [ ]:
# Load sentence test set and compute alpha from formula using checkpoints source table
sent_df = pd.read_csv(DATA_PATH)
if "id" not in sent_df.columns or "question" not in sent_df.columns:
    raise ValueError("Sentence test.csv must contain columns: id, question")

alpha_src_df = pd.read_csv(ALPHA_SOURCE_PATH)
required_alpha_cols = {"id", "verbal_uncertainty", "sentence_semantic_entropy"}
missing = required_alpha_cols - set(alpha_src_df.columns)
if missing:
    raise ValueError(f"Alpha source file missing columns: {missing}")

alpha_src_df = alpha_src_df[["id", "verbal_uncertainty", "sentence_semantic_entropy"]].copy()
alpha_src_df["alpha_raw"] = (
    alpha_src_df["sentence_semantic_entropy"].to_numpy() / MAX_SE
    - alpha_src_df["verbal_uncertainty"].to_numpy()
) * MAX_ALPHA
alpha_src_df["alpha"] = alpha_src_df["alpha_raw"].clip(0, MAX_ALPHA)

if USE_ALPHA_QUANTIZATION:
    alpha_src_df["alpha"] = (
        (alpha_src_df["alpha"] / ALPHA_QUANT_STEP).round() * ALPHA_QUANT_STEP
    ).clip(0, MAX_ALPHA)

alpha_src_df["alpha"] = alpha_src_df["alpha"].round(4)

df = sent_df.merge(alpha_src_df[["id", "alpha"]], on="id", how="left")
if df["alpha"].isna().any():
    missing_ids = df.loc[df["alpha"].isna(), "id"].head(10).tolist()
    raise ValueError(f"No alpha for some ids (first 10): {missing_ids}")

print("Rows:", len(df))
print("Alpha stats:", {
    "min": float(df["alpha"].min()),
    "max": float(df["alpha"].max()),
    "mean": float(df["alpha"].mean()),
    "unique": int(df["alpha"].nunique()),
})
print(df[["id", "question", "alpha"]].head(3))


Rows: 500
Alpha stats: {'min': 0.0, 'max': 1.6, 'mean': 0.5624000000000001, 'unique': 5}
       id                                           question  alpha
0  test_0     where does the optic nerve cross the midline ​    0.0
1  test_1  when does walking dead season 8 second half start    0.8
2  test_2       who was the chief guest of 2014 republic day    0.8


In [ ]:
# Load hedge vector
hedge = torch.load(HEDGE_PATH, map_location="cpu")
if hedge.ndim != 2:
    raise ValueError(f"Expected [n_layers, d_model], got {tuple(hedge.shape)}")
for l in LAYER_IDS:
    if l >= hedge.shape[0]:
        raise ValueError(f"Layer {l} out of range for hedge n_layers={hedge.shape[0]}")

print("hedge shape:", tuple(hedge.shape))


hedge shape: (32, 4096)


In [ ]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded


In [ ]:
def clear_hooks(model):
    if not hasattr(model, "_vuf_hooks"):
        return
    for h in model._vuf_hooks:
        h.remove()
    model._vuf_hooks = []


def register_intervention_hooks(model, hedge_2d, layer_ids, alpha):
    clear_hooks(model)
    model._vuf_hooks = []

    for layer_id in layer_ids:
        direction = hedge_2d[layer_id].float()
        direction = direction / (direction.norm() + 1e-8)

        layer_mod = model.model.layers[layer_id]
        p = next(layer_mod.parameters())
        h_vec = direction.to(device=p.device, dtype=p.dtype)
        alpha_f = float(alpha)

        def make_hook(vec, alpha_local):
            def hook_fn(module, inputs, outputs):
                if torch.is_tensor(outputs):
                    out = outputs + vec * alpha_local
                    return out
                if isinstance(outputs, tuple):
                    out0 = outputs[0] + vec * alpha_local
                    return (out0,) + outputs[1:]
                raise TypeError(f"Unexpected outputs type: {type(outputs)}")
            return hook_fn

        handle = layer_mod.register_forward_hook(make_hook(h_vec, alpha_f))
        model._vuf_hooks.append(handle)


def build_prompt(question):
    messages = [
        {"role": "user", "content": f"Question: {question}\nAnswer:"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch_original_format(questions, n_responses=10, max_new_tokens=MAX_NEW_TOKENS):
    prompts = [build_prompt(q) for q in questions]
    inputs = tokenizer(
        prompts,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.1,
        )
        outputs_responses = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            num_return_sequences=n_responses,
        )

    decoded_answers = tokenizer.batch_decode(
        outputs[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    decoded_responses = tokenizer.batch_decode(
        outputs_responses[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    assert len(decoded_answers) == len(questions)
    assert len(decoded_responses) == len(questions) * n_responses

    per_q_responses = [
        decoded_responses[i * n_responses : (i + 1) * n_responses]
        for i in range(len(questions))
    ]
    return decoded_answers, per_q_responses


In [ ]:
# Smoke test (same output schema as original script)
smoke_df = df.head(6).copy()

if SMOKE_OUT.exists():
    SMOKE_OUT.unlink()

records = smoke_df[["question", "alpha"]].to_dict("records")
with SMOKE_OUT.open("w", encoding="utf-8") as fout:
    for start in tqdm(range(0, len(records), SMOKE_BATCH_SIZE)):
        chunk = records[start : start + SMOKE_BATCH_SIZE]
        grouped = defaultdict(list)
        for r in chunk:
            grouped[float(r["alpha"])].append(r)

        for a, items in grouped.items():
            qs = [it["question"] for it in items]

            clear_hooks(model)
            if a > 0:
                register_intervention_hooks(model, hedge, LAYER_IDS, alpha=a)
            most_likely, responses = generate_batch_original_format(qs, n_responses=10)

            for i, q in enumerate(qs):
                line = {
                    "alpha": float(a),
                    "question": q,
                    "most_likely_answer": most_likely[i],
                    "responses": responses[i],
                }
                fout.write(json.dumps(line, ensure_ascii=False) + "\n")

clear_hooks(model)
print(f"Smoke results saved to: {SMOKE_OUT}")


  0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Smoke results saved to: /checkpoints/intervention_outputs/smoke_sentence_layers15_23_q0p4.jsonl


In [ ]:
# Full run (resume-safe, same output schema as original script)
processed = 0
if FULL_OUT.exists():
    with FULL_OUT.open("r", encoding="utf-8") as f:
        processed = sum(1 for _ in f)

print("Already processed:", processed)
records = df.iloc[processed:][["question", "alpha"]].to_dict("records")

mode = "a" if FULL_OUT.exists() else "w"
with FULL_OUT.open(mode, encoding="utf-8") as fout:
    for start in tqdm(range(0, len(records), FULL_BATCH_SIZE)):
        chunk = records[start : start + FULL_BATCH_SIZE]
        grouped = defaultdict(list)
        for r in chunk:
            grouped[float(r["alpha"])].append(r)

        for a, items in grouped.items():
            qs = [it["question"] for it in items]

            clear_hooks(model)
            if a > 0:
                register_intervention_hooks(model, hedge, LAYER_IDS, alpha=a)
            most_likely, responses = generate_batch_original_format(qs, n_responses=10)

            for i, q in enumerate(qs):
                line = {
                    "alpha": float(a),
                    "question": q,
                    "most_likely_answer": most_likely[i],
                    "responses": responses[i],
                }
                fout.write(json.dumps(line, ensure_ascii=False) + "\n")

clear_hooks(model)
print(f"Full run saved to: {FULL_OUT}")


Already processed: 0


  0%|          | 0/13 [00:00<?, ?it/s]

Full run saved to: /checkpoints/intervention_outputs/full_sentence_layers15_23_q0p4.jsonl
